# 05 – S5 (cGAN tanpa ROI) dan S6 (cGAN berpandu ROI) (CP5, FR-06, FR-07)

Satu GAN per lipatan membangkitkan 100% jumlah citra ganas latih sekali; subset 25%/50% diambil darinya. Rantai per lipatan: GAN → sintetis → Swin-T ×3 rasio (urutan wajib). Rasio terbaik dipilih per lipatan dari **validasi**, bukan uji (G2, L4). Semua rasio dilaporkan.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

def _root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'configs' / 'base.yaml').exists():
            return q
    raise FileNotFoundError('configs/base.yaml tidak ditemukan')
ROOT = _root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src.config import load_merged, P
from src.launch import run_module

CFG = 'configs/base.yaml'
cfg = load_merged(CFG)
DEVICES = None      # mis. [0, 1]: satu rantai job per GPU lewat CUDA_VISIBLE_DEVICES (PRD §8)
REPS, FOLDS = None, None   # None = semua. Membatasi JOB YANG DILUNCURKAN saja, bukan memilih hasil; gerbang butuh semuanya.

from src.jobs import launch_gan_chain
from src.evaluate import evaluate_all_ratios, evaluate_selected, select_best_config, compare_scenarios, load_result

cfg5, cfg6 = load_merged('configs/s5.yaml'), load_merged('configs/s6.yaml')
ALLOW_PARTIAL = REPS is not None or FOLDS is not None

## 1. S5 – 10 rantai (GAN λ=0 → 3 rasio)

In [ ]:
launch_gan_chain('configs/s5.yaml', reps=REPS, folds=FOLDS, devices=DEVICES)

## 2. S6 – 10 rantai (GAN berpandu ROI → 3 rasio). λ_roi dari CP4.

In [ ]:
launch_gan_chain('configs/s6.yaml', reps=REPS, folds=FOLDS, devices=DEVICES)

## 3. Evaluasi: semua rasio, dan rasio terpilih per lipatan dari validasi

In [ ]:
for base, c in (('S5', cfg5), ('S6', cfg6)):
    display(evaluate_all_ratios(c, base, ALLOW_PARTIAL))
    r = evaluate_selected(c, base, ALLOW_PARTIAL)
    print(base, 'rasio terpilih (validasi):')
    display(r['table'])
    display(pd.read_csv(P(c, 'results', f'ratio_selection_{base}.csv')).head(10))

## 4. Konfigurasi terbaik untuk S7 (dari validasi) dan ablation ROI S6 − S5 (**eksploratif**)

In [ ]:
if not ALLOW_PARTIAL:
    print(json.dumps(select_best_config(cfg6), indent=2))
display(compare_scenarios(cfg6, load_result(cfg6, 'S6'), load_result(cfg5, 'S5')))

## 5. Pemeriksaan checkpoint CP5 dan `manifest.json` (sel terakhir)

Periksa `grid_test/samples_malignant.png` dan LPIPS tiap lipatan untuk mode collapse (`python -m src.approve gan-visual CP5 --by NAMA`).

In [ ]:
from src.checkpoint import finalize_cp5
report = finalize_cp5(cfg5, cfg6)
assert report['checkpoint'] == 'CP5'